# Spotify Track Metadata | Exploratory Analysis
**Objective:** Describe explicit-content labels, track durations, and release dates in a small Spotify search sample.

The sample contains up to 10 tracks for each of five search terms. These are **search-result groups, not verified musical genres**, and the results are not representative of Spotify's full catalog. Popularity is not evaluated because it was unavailable in the original API responses.

## 1. Setup and data source
By default, the notebook loads `../data/spotify_tracks.csv` from the portfolio repository. If the CSV is missing, it can collect fresh data using Spotify credentials stored in environment variables (`SPOTIFY_CLIENT_ID` and `SPOTIFY_CLIENT_SECRET`). Never paste credentials into this notebook or commit a `.env` file.

Install dependencies if needed: `pip install pandas matplotlib requests`.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt

SEARCH_TERMS = ["pop", "hip hop", "country", "rock", "electronic"]
DATA_CANDIDATES = [Path("../data/spotify_tracks.csv"), Path("data/spotify_tracks.csv"), Path("spotify_tracks.csv")]
OUTPUT_DIR = Path("../assets") if Path("../assets").is_dir() else Path("assets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def collect_spotify_tracks():
    import requests
    client_id = os.getenv("SPOTIFY_CLIENT_ID")
    client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")
    if not client_id or not client_secret:
        raise RuntimeError("CSV not found. Set SPOTIFY_CLIENT_ID and SPOTIFY_CLIENT_SECRET to collect fresh data.")
    auth = requests.post(
        "https://accounts.spotify.com/api/token",
        data={"grant_type": "client_credentials"},
        auth=(client_id, client_secret),
        timeout=30,
    )
    auth.raise_for_status()
    token = auth.json()["access_token"]
    session = requests.Session()
    session.headers.update({"Authorization": f"Bearer {token}"})
    records = []
    for term in SEARCH_TERMS:
        response = session.get(
            "https://api.spotify.com/v1/search",
            params={"q": term, "type": "track", "limit": 10},
            timeout=30,
        )
        response.raise_for_status()
        for track in response.json().get("tracks", {}).get("items", []):
            if not track:
                continue
            artists = track.get("artists") or []
            records.append({
                "track_name": track.get("name"),
                "artist": artists[0].get("name") if artists else None,
                "search_term": term,
                "explicit": track.get("explicit"),
                "duration_ms": track.get("duration_ms"),
                "release_date": (track.get("album") or {}).get("release_date"),
            })
    return pd.DataFrame.from_records(records)

csv_path = next((p for p in DATA_CANDIDATES if p.is_file()), None)
if csv_path:
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} rows from {csv_path}")
else:
    df = collect_spotify_tracks()
    df.to_csv(Path("spotify_tracks.csv"), index=False)
    print(f"Collected {len(df)} tracks; saved spotify_tracks.csv")

required = {"track_name", "artist", "search_term", "explicit", "duration_ms", "release_date"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required CSV columns: {sorted(missing)}")

df["duration_ms"] = pd.to_numeric(df["duration_ms"], errors="coerce")
df["explicit"] = df["explicit"].astype("string").str.strip().str.lower().map(
    {"true": True, "false": False, "1": True, "0": False}
)
df["release_date_parsed"] = pd.to_datetime(df["release_date"], errors="coerce", format="mixed")
df["duration_min"] = df["duration_ms"] / 60000
df = df.dropna(subset=["search_term", "duration_min", "explicit"])
df = df[df["duration_min"] > 0].copy()
if df.empty:
    raise ValueError("No usable track records after validation.")
print(f"Usable rows: {len(df)}; search groups: {df['search_term'].nunique()}")
display(df.head())

## 2. Descriptive summary
These summaries describe the collected records only. Search terms can overlap, and tracks may appear in more than one group.

In [ ]:
summary = df.groupby("search_term", sort=False).agg(
    tracks=("track_name", "size"),
    explicit_tracks=("explicit", "sum"),
    median_duration_min=("duration_min", "median"),
)
summary["explicit_share_pct"] = 100 * summary["explicit_tracks"] / summary["tracks"]
display(summary.round(2))
print(f"Overall explicit share: {100 * df['explicit'].mean():.1f}%")

## 3. Visualizations
Charts are saved to the repository's `assets` folder (or a local `assets` folder when running elsewhere).

In [ ]:
groups = [g for g in SEARCH_TERMS if g in df["search_term"].unique()]
clean = [df.loc[(df["search_term"] == g) & (df["explicit"] == False), "duration_min"].values for g in groups]
explicit = [df.loc[(df["search_term"] == g) & (df["explicit"] == True), "duration_min"].values for g in groups]

fig, ax = plt.subplots(figsize=(10, 5))
positions, values, colors = [], [], []
for i, (a, b) in enumerate(zip(clean, explicit)):
    if len(a):
        positions.append(i * 3); values.append(a); colors.append("#3b82f6")
    if len(b):
        positions.append(i * 3 + 1); values.append(b); colors.append("#e87979")
if values:
    boxes = ax.boxplot(values, positions=positions, widths=0.75, patch_artist=True)
    for patch, color in zip(boxes["boxes"], colors):
        patch.set_facecolor(color)
ax.set_xticks([i * 3 + 0.5 for i in range(len(groups))], groups)
ax.set_ylabel("Track duration (minutes)")
ax.set_title("Track duration by search term and explicit label")
ax.plot([], [], color="#3b82f6", linewidth=8, label="Not explicit")
ax.plot([], [], color="#e87979", linewidth=8, label="Explicit")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "spotify-duration.png", dpi=170)
plt.show()

In [ ]:
shares = summary["explicit_share_pct"].reindex(groups)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(shares.index, shares.values, color="#3b82f6")
ax.set_ylim(0, 100)
ax.set_ylabel("Explicit-labeled tracks (%)")
ax.set_title("Explicit-content labels by search term")
for i, value in enumerate(shares.values):
    ax.text(i, value + 2, f"{value:.0f}%", ha="center")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "spotify-explicit-share.png", dpi=170)
plt.show()

In [ ]:
dated = df.dropna(subset=["release_date_parsed"])
fig, ax = plt.subplots(figsize=(10, 5))
for term, group in dated.groupby("search_term"):
    ax.scatter(group["release_date_parsed"], group["duration_min"], label=term, alpha=0.75)
ax.set_title("Track duration and release date in the search sample")
ax.set_xlabel("Release date")
ax.set_ylabel("Track duration (minutes)")
ax.legend(title="Search term", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "spotify-release-dates.png", dpi=170)
plt.show()

## 4. Takeaways and limitations
- This notebook documents API collection, data validation, descriptive summaries, and visualizations.
- Differences between search-result groups are **descriptive**, not evidence of differences across entire genres.
- The sample is small and nonrandom; search rankings and repeated tracks can affect the patterns.
- Spotify's explicit flag is a platform metadata label, not a direct assessment of lyrics.
- **No popularity conclusions are supported** because the original sample did not include usable popularity scores.

**Next step:** Collect a larger, documented sample and evaluate duplicate tracks and sampling effects before drawing broader conclusions.